# Data Farming with Categorical (Enum) Factors

This notebook demonstrates how to run a data farming experiment when one of the factors is a categorical variable, specifically an enumeration (`Enum`) type.

The `simopt.data_farming.create_design` function expects numerical values for all factors. To work around this for categorical factors, we can:
1.  **Map the Enum to Integers**: Create a mapping from each enum member to a unique integer.
2.  **Generate the Design**: Use these integers to define the levels of the factor in the experimental design.
3.  **Define an Evaluation Function**: Create a wrapper function that takes a design point, looks up the original enum member from the integer, and then runs the simulation.
4.  **Run the Experiment**: Execute the data farming run.
5.  **Post-process Results**: Map the integer values in the results back to the original enum member names for easier analysis.

We will demonstrate this by evaluating the performance of the `ASTROMoRF` solver with different `PolyBasisType` options on the `SSCONT` problem.

## 1. Import Required Libraries

In [1]:
import sys
from pathlib import Path
from typing import cast

# Take the current directory, find the parent, and add it to the system path
sys.path.append(str(Path.cwd().parent))

In [2]:
import pandas as pd

from simopt.experiment_base import ProblemsSolvers
from simopt.solvers.astromorf import PolyBasisType

POSTREP_METRIC_LABEL = "post-replication objective mean"
FALLBACK_METRIC_LABEL = "in-solve objective estimate"

# Initialize result tables so analysis cells are safe before running experiments.
recommended_solns = pd.DataFrame(
    columns=cast(pd.Index,
                 [
                     "solver",
                     "problem",
                     "macrorep",
                     "step",
                     "budget", 
                     "solution"
                ]
    )
)
all_responses = pd.DataFrame(
    columns=cast(pd.Index,
                 [
                     "solver",
                     "problem",
                     "macrorep",
                     "iteration",
                     "budget",
                     "response_value",
                     "response_metric",
                ]
    )
)
all_postreps = pd.DataFrame(
    columns=cast(pd.Index,
                 [
                     "solver",
                     "problem",
                     "macrorep",
                     "budget_index",
                     "budget",
                     "postrep",
                     "response_value",
                ]
    )
)
all_postreps_init_opt = pd.DataFrame(
    columns=cast(pd.Index,
                 [
                     "solver",
                     "problem",
                     "macrorep",
                     "solution_type",
                     "response_value"
                 ]
            )
)


def _solver_label(problem_solver):
    basis = problem_solver.solver.factors.get("polynomial basis")
    if basis is None:
        basis = problem_solver.solver.factors.get("polynomial_basis")
    basis_name = basis.name if hasattr(basis, "name") else str(basis)
    return f"{problem_solver.solver.name}:{basis_name}"


def _build_result_tables(group_experiment: ProblemsSolvers):
    recommended_rows = []
    response_rows = []
    postrep_rows = []
    init_opt_rows = []

    for solver_group in group_experiment.experiments:
        for problem_solver in solver_group:
            solver_label = _solver_label(problem_solver)
            problem_name = problem_solver.problem.name

            all_budgets = getattr(problem_solver, "all_intermediate_budgets", []) or []
            all_solutions = getattr(problem_solver, "all_recommended_xs", []) or []

            for mrep, (budgets, solutions) in enumerate(zip(all_budgets, all_solutions)):
                for step, (budget, solution) in enumerate(zip(budgets, solutions)):
                    recommended_rows.append(
                        {
                            "solver": solver_label,
                            "problem": problem_name,
                            "macrorep": mrep,
                            "step": step,
                            "budget": budget,
                            "solution": solution,
                        }
                    )

            if getattr(problem_solver, "has_postreplicated", False):
                all_est_objectives = getattr(problem_solver, "all_est_objectives", []) or []
                for mrep, (budgets, est_objectives) in enumerate(
                    zip(all_budgets, all_est_objectives)
                ):
                    for iteration, (budget, response) in enumerate(
                        zip(budgets, est_objectives)
                    ):
                        if pd.notna(response):
                            response_rows.append(
                                {
                                    "solver": solver_label,
                                    "problem": problem_name,
                                    "macrorep": mrep,
                                    "iteration": iteration,
                                    "budget": budget,
                                    "response_value": float(response),
                                    "response_metric": POSTREP_METRIC_LABEL,
                                }
                            )

                for mrep, per_budget_postreps in enumerate(problem_solver.all_post_replicates):
                    budgets = all_budgets[mrep] if mrep < len(all_budgets) else []
                    for budget_index, reps in enumerate(per_budget_postreps):
                        budget = budgets[budget_index] if budget_index < len(budgets) else None
                        for postrep, response in enumerate(reps):
                            postrep_rows.append(
                                {
                                    "solver": solver_label,
                                    "problem": problem_name,
                                    "macrorep": mrep,
                                    "budget_index": budget_index,
                                    "budget": budget,
                                    "postrep": postrep,
                                    "response_value": float(response),
                                }
                            )
            else:
                all_budget_history = getattr(problem_solver, "all_budget_history", None)
                all_fn_estimates = getattr(problem_solver, "all_fn_estimates", None)
                if all_budget_history and all_fn_estimates:
                    for mrep, (budget_history, fn_estimates) in enumerate(
                        zip(all_budget_history, all_fn_estimates)
                    ):
                        for iteration, (budget, response) in enumerate(
                            zip(budget_history, fn_estimates)
                        ):
                            if pd.notna(response):
                                response_rows.append(
                                    {
                                        "solver": solver_label,
                                        "problem": problem_name,
                                        "macrorep": mrep,
                                        "iteration": iteration,
                                        "budget": budget,
                                        "response_value": float(response),
                                        "response_metric": FALLBACK_METRIC_LABEL,
                                    }
                                )

            x0_postreps = getattr(problem_solver, "x0_postreps", []) or []
            xstar_postreps = getattr(problem_solver, "xstar_postreps", []) or []
            for mrep, response in enumerate(x0_postreps):
                init_opt_rows.append(
                    {
                        "solver": solver_label,
                        "problem": problem_name,
                        "macrorep": mrep,
                        "solution_type": "x0",
                        "response_value": float(response),
                    }
                )
            for mrep, response in enumerate(xstar_postreps):
                init_opt_rows.append(
                    {
                        "solver": solver_label,
                        "problem": problem_name,
                        "macrorep": mrep,
                        "solution_type": "xstar",
                        "response_value": float(response),
                    }
                )

    return (
        pd.DataFrame(recommended_rows, columns=recommended_solns.columns),
        pd.DataFrame(response_rows, columns=all_responses.columns),
        pd.DataFrame(postrep_rows, columns=all_postreps.columns),
        pd.DataFrame(init_opt_rows, columns=all_postreps_init_opt.columns),
    )

## 2. Create Enum-to-Integer Mapping

In [3]:
# Create a list of the enum members
poly_basis_options = list(PolyBasisType)

# Create a mapping from the enum member to an integer
poly_basis_to_int = {basis: i for i, basis in enumerate(poly_basis_options)}

# Create a reverse mapping from the integer to the enum member
int_to_poly_basis = {i: basis for i, basis in enumerate(poly_basis_options)}

print("Enum to Integer Mapping:")
print(poly_basis_to_int)
print("\nInteger to Enum Mapping:")
print(int_to_poly_basis)

Enum to Integer Mapping:
{<PolyBasisType.HERMITE: 'hermite'>: 0, <PolyBasisType.LEGENDRE: 'legendre'>: 1, <PolyBasisType.CHEBYSHEV: 'chebyshev'>: 2, <PolyBasisType.MONOMIAL: 'monomial'>: 3, <PolyBasisType.NATURAL: 'natural'>: 4, <PolyBasisType.MONOMIAL_POLY: 'monomial_polynomial'>: 5, <PolyBasisType.LAGRANGE: 'lagrange'>: 6, <PolyBasisType.NFP: 'nfp'>: 7, <PolyBasisType.LAGUERRE: 'laguerre'>: 8}

Integer to Enum Mapping:
{0: <PolyBasisType.HERMITE: 'hermite'>, 1: <PolyBasisType.LEGENDRE: 'legendre'>, 2: <PolyBasisType.CHEBYSHEV: 'chebyshev'>, 3: <PolyBasisType.MONOMIAL: 'monomial'>, 4: <PolyBasisType.NATURAL: 'natural'>, 5: <PolyBasisType.MONOMIAL_POLY: 'monomial_polynomial'>, 6: <PolyBasisType.LAGRANGE: 'lagrange'>, 7: <PolyBasisType.NFP: 'nfp'>, 8: <PolyBasisType.LAGUERRE: 'laguerre'>}


## 3. Generate the Experimental Design

In [4]:
from IPython.display import display

# Define the factors for the experiment.
# The `polynomial basis` factor uses integer representations.
factor_settings = {
    "polynomial basis": list(int_to_poly_basis.keys())
}

# Build a simple explicit design table for this notebook.
design = [
    {"polynomial basis": basis}
    for basis in factor_settings["polynomial basis"]
]

print("Experimental Design:")
display(design)

Experimental Design:


[{'polynomial basis': 0},
 {'polynomial basis': 1},
 {'polynomial basis': 2},
 {'polynomial basis': 3},
 {'polynomial basis': 4},
 {'polynomial basis': 5},
 {'polynomial basis': 6},
 {'polynomial basis': 7},
 {'polynomial basis': 8}]

## 4. Define the Evaluation Function

In [5]:
from simopt.experiment_base import ProblemSolver
# Create a list-of-lists of ProblemSolver objects for ProblemsSolvers.
# Outer list: solver configurations; inner list: problems.
experiments = []
for basis_enum in int_to_poly_basis.values():
    experiments.append(
        [
            ProblemSolver(
                solver_name="ASTROMORF",
                problem_name="SSCONT-1",
                solver_fixed_factors={"polynomial basis": basis_enum},
                problem_fixed_factors={"budget": 200},
            )
        ]
    )

## 5. Run the Datafarming Experiment

In [6]:
# Create ProblemsSolvers experiment
experiment = ProblemsSolvers(experiments=experiments)

# check compatibility of selected solvers and problems
experiment.check_compatibility()

''

## 6. Analyze the Results

In [7]:
n_macroreps = 3
n_postreps = 50

# Run experiment and generate post-replication objective estimates.
experiment.run(n_macroreps=n_macroreps)
experiment.post_replicate(n_postreps=n_postreps)

(
    recommended_solns,
    all_responses,
    all_postreps,
    all_postreps_init_opt,
) = _build_result_tables(experiment)

print(
    "Experiment finished. "
    f"n_macroreps={n_macroreps}, "
    f"n_postreps={n_postreps}, "
    f"recommended_solns={len(recommended_solns)}, "
    f"all_responses={len(all_responses)}, "
    f"all_postreps={len(all_postreps)}, "
    f"all_postreps_init_opt={len(all_postreps_init_opt)}."
)

Experiment finished. n_macroreps=3, n_postreps=50, recommended_solns=150, all_responses=150, all_postreps=7500, all_postreps_init_opt=0.


In [8]:
from IPython.display import display

if recommended_solns.empty:
    print("No recommended solutions yet. Run the experiment cell below.")
else:
    print("Recommended Solutions:")
    display(recommended_solns)

if all_responses.empty:
    print("No response rows available yet. Run the experiment and post-replication cell below.")
else:
    print("All Responses:")
    display(all_responses)

    metric_name = "response"
    if "response_metric" in all_responses and not all_responses["response_metric"].empty:
        metric_name = all_responses["response_metric"].mode(dropna=True).iat[0]

    print(f"\nAverage {metric_name} by basis type:")
    avg_responses = all_responses.groupby("solver")["response_value"].mean().sort_values()
    display(avg_responses)

Recommended Solutions:


,solver,problem,macrorep,step,budget,solution
0,ASTROMORF:HERMITE,SSCONT-1,0,0,5,"(300, 300)"
1,ASTROMORF:HERMITE,SSCONT-1,0,1,20,"(277.63932100764475, 277.63932100764475)"
2,ASTROMORF:HERMITE,SSCONT-1,0,2,86,"(259.92718154930947, 259.92718154930947)"
3,ASTROMORF:HERMITE,SSCONT-1,0,3,200,"(259.92718154930947, 259.92718154930947)"
4,ASTROMORF:HERMITE,SSCONT-1,1,0,5,"(300, 300)"
...,...,...,...,...,...,...
145,ASTROMORF:LAGUERRE,SSCONT-1,2,1,20,"(287.8099716267191, 287.8099716267191)"
146,ASTROMORF:LAGUERRE,SSCONT-1,2,2,40,"(291.61507913905285, 291.61511457857154)"
147,ASTROMORF:LAGUERRE,SSCONT-1,2,3,60,"(288.92616663658896, 288.9257989254234)"
148,ASTROMORF:LAGUERRE,SSCONT-1,2,4,87,"(292.5590735581171, 292.5590434520811)"


All Responses:


,solver,problem,macrorep,iteration,budget,response_value,response_metric
0,ASTROMORF:HERMITE,SSCONT-1,0,0,5,527.122721,post-replication objective mean
1,ASTROMORF:HERMITE,SSCONT-1,0,1,20,534.709362,post-replication objective mean
2,ASTROMORF:HERMITE,SSCONT-1,0,2,86,538.568206,post-replication objective mean
3,ASTROMORF:HERMITE,SSCONT-1,0,3,200,538.568206,post-replication objective mean
4,ASTROMORF:HERMITE,SSCONT-1,1,0,5,520.057579,post-replication objective mean
...,...,...,...,...,...,...,...
145,ASTROMORF:LAGUERRE,SSCONT-1,2,1,20,544.794965,post-replication objective mean
146,ASTROMORF:LAGUERRE,SSCONT-1,2,2,40,544.270970,post-replication objective mean
147,ASTROMORF:LAGUERRE,SSCONT-1,2,3,60,544.686903,post-replication objective mean
148,ASTROMORF:LAGUERRE,SSCONT-1,2,4,87,544.286582,post-replication objective mean



Average post-replication objective mean by basis type:


solver
ASTROMORF:LEGENDRE         525.556171
ASTROMORF:MONOMIAL         527.740589
ASTROMORF:MONOMIAL_POLY    528.194909
ASTROMORF:LAGRANGE         528.241304
ASTROMORF:LAGUERRE         528.241304
ASTROMORF:CHEBYSHEV        528.455911
ASTROMORF:NFP              528.639706
ASTROMORF:NATURAL          530.357221
ASTROMORF:HERMITE          534.843283
Name: response_value, dtype: float64